# Scorers and LLM Judges in MLflow's GenAI Evaluation

[MLflow 3.14.0's source code](https://github.com/mlflow/mlflow/tree/v3.14.0)

This is the 2nd part in the [GenAI Evaluation Framework in MLflow 3.14]($./00_GenAI Evaluation Framework in MLflow 3.14) series.

# Set Up Environment

In [0]:
%pip install -qU "mlflow-skinny[databricks]>=3.14.0" "databricks-agents>=1.11.0"
%restart_python


Just FYI that `mlflow-skinny`'s [databricks](https://github.com/mlflow/mlflow/blob/v3.14.0/pyproject.toml#L89-L95) optional dependency group pulls the following dependencies:

<br>

```py
databricks = [
  "azure-storage-file-datalake>12",
  "google-cloud-storage>=1.30.0",
  "boto3>1",
  "botocore",
  "databricks-agents>=1.2.0,<2.0",
]
```


Among the optional dependencies is `databricks-agents` that we explicitly upgraded to the [latest version](https://pypi.org/project/databricks-agents/).

In [0]:
%pip show databricks-agents

In [0]:
import mlflow

displayHTML(f"MLflow version: <b>{mlflow.__version__}</b>")

# Scorers: What Are They Actually?

## Scorers as Core Component of GenAI Evaluation in MLflow


Scorers are the core components of [MLflow's GenAI Evaluation harness](https://mlflow.org/docs/latest/genai/eval-monitor/) (i.e., [mlflow.genai.evaluate](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html#mlflow.genai.evaluate)).

<br>

```
evaluate(
    data: 'EvaluationDatasetTypes',
    scorers: list[mlflow.genai.scorers.base.Scorer], // 👈 here
    predict_fn: Optional[Callable[..., Any]] = None,
    model_id: str | None = None
) -> 'EvaluationResult'
```

Every scorer in `scorers` is evaluated on every trace record in `data` (and optionally on `predict_fn` that is your GenAI application).


## Scorers are Pydantic Models

**Scorer** is a Python class [mlflow.genai.scorers.base.Scorer](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/base.py#L218-L1135) which is also a Pydantic model (extends [pydantic.BaseModel](https://pydantic.dev/docs/validation/latest/concepts/models/)).

<br>

```py
class Scorer(BaseModel):
    ...
```

As Pydantic models, Scorers can be serialized to and deserialized from JSON representation.


All scorers have the following properties.

Name | Type | Description
-|-|-
`name` | `str` | (required)
`aggregations` | `list[_AggregationType]` | (optional) Discussed later in this notebook.
`description` | `str` | (optional)
`kind` | [ScorerKind](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/base.py#L47-L54) enum |
`is_session_level_scorer` | `bool` | Default: `False`. Discussed later in this notebook.

There are a couple of private attributes (`PrivateAttr`s).


There are [built-in](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/__init__.py) (provided by MLflow) and custom scorers.

🚨 [Report this "discrepancy"](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/__init__.py#L124): `Equivalence` is not alphanumerically ordered.

<br>

```py
from mlflow.genai.scorers import Correctness, Safety
```

All the built-in scorers inherit from [BuiltInScorer(Judge)](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L313C1-L388) class.

[Judge](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/judges/base.py#L53C1-L137) class is a...[Scorer](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/base.py#L218-L1120).

You can invoke scorers directly with a single input (for testing), or pass them to `mlflow.genai.evaluate` for running full evaluation on a dataset.

In [0]:
from mlflow.genai.scorers import get_all_scorers

scorers = get_all_scorers()
print(scorers)
print(len(scorers))

## Scorers are Functions (Callables)

<br>

```py
def __call__(
    self,
    *,
    inputs: Any = None,
    outputs: Any = None,
    expectations: dict[str, Any] | None = None,
    trace: Trace | None = None,
    session: list[Trace] | None = None,
) -> int | float | bool | str | Feedback | list[Feedback]:
    raise NotImplementedError("Implementation of __call__ is required for Scorer class")
```

Scorers raise a `NotImplementedError` by default.

User-defined scorers don't need to have all the parameters defined from the base signature.
You can define a custom scorer with only the parameters you need.

When executed, scorers return one of the following:

* `int`
* `float`
* `bool`
* `str`
* `mlflow.entities.assessment.Feedback` (discussed later)
* `list[mlflow.entities.assessment.Feedback]`

In [0]:
from mlflow.genai import Scorer

help(Scorer.__call__)


### Automatic Trace Evaluation

Scorers can be registered with a MLflow server (`ScorerStoreRegistry`) for automatic trace evaluation in the specified experiment (using `Scorer.register` method).

<br>

```py
def register(
    self,
    *,
    name: str | None = None,
    experiment_id: str | None = None) -> "Scorer":
    ...
```

`experiment_id` is the ID of the MLflow experiment to register the scorer for.
If undefined (`None`), uses the currently active experiment.

Once registered, the scorer can be `start`ed to begin evaluating traces automatically.

The scorer registration can be `update`d and eventually `stop`ped (deactivated).

In [0]:
from mlflow.genai.scorers import Correctness

help(Correctness.register)

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery

# The same name must not be used twice
# hence try-except block for reproducibility

scorer_name = "relevance_scorer"
try:
    registered_scorer = RelevanceToQuery().register(name=scorer_name)
    print(registered_scorer)
except ValueError as ve:
    print(f"Exception: {ve}")

In [0]:
# The current notebook's experiment can be queried using this method
# but https://github.com/mlflow/mlflow/issues/24387

from mlflow.tracking.fluent import _get_experiment_id

current_notebook_experiment_id = _get_experiment_id()

In [0]:
from mlflow.genai.scorers import scorer

@scorer
def custom_length_check(outputs) -> bool:
    return len(outputs) > 100

# The same name must not be used twice
# hence try-except block for reproducibility

custom_scorer_name = "output_length_checker"
try:
    registered_custom_scorer = custom_length_check.register(
        name=custom_scorer_name,
        experiment_id=current_notebook_experiment_id,
    )
    print(registered_custom_scorer)
except ValueError as ve:
    print(f"Exception: {ve}")

In [0]:
custom_length_check


### Mosaic AI Agent Framework SDK

In Databricks workspace, the [databricks-agents](https://pypi.org/project/databricks-agents/) package is required to register scorers.

In [0]:
%pip show databricks-agents

In [0]:
import mlflow

# Get the current tracking uri
tracking_uri = mlflow.get_tracking_uri()
print(f"Current tracking uri: {tracking_uri}")

assert tracking_uri == "databricks"

# Get the scorer store for the tracking uri
scorer_store = mlflow.genai.scorers.registry._get_scorer_store(tracking_uri=tracking_uri)
print(f"Scorer store: {scorer_store}")

registered_scorer = scorer_store.get_scorer(name=scorer_name, experiment_id=current_notebook_experiment_id)
print(f"Registered scorer: {registered_scorer}")

In [0]:
# All registered scorers in Databricks will have databricks backend assigned

from mlflow.genai.scorers.base import SCORER_BACKEND_DATABRICKS

print(SCORER_BACKEND_DATABRICKS)


## Built-In Scorers

**Built-in Scorers** inherit from [BuiltInScorer](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L314-L389):

* [Correctness](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1699-L1891) evaluates whether the model's response supports the expected facts or response.
* [Safety](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1593-L1695) ensures that the agent's responses do not contain harmful, offensive, or toxic content.
* [a few others](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/__init__.py#L30-L53)

Properties:
* `name`
* `required_columns`
* `inference_params` (optional)
* `instructions` of what this scorer evaluates

Operators:
* `model_dump` for serialization (using `pydantic.BaseModel.model_dump` in JSON mode)
* `model_validate` for deserialization

In [0]:
import pydantic

print(pydantic.__version__)

In [0]:
import pydantic

help(pydantic.BaseModel.model_dump)

## Examples

`expectations` field can only be a dict with `expected_response` or `expected_facts`.

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

scorer = Correctness(name="my_correctness")

print(f"description:\n{scorer.description}")
print("")
print(f"instructions:\n{scorer.instructions}")

In [0]:
# Scorers are callables
assessment = scorer(
    # required
    # https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1771
    inputs={
        "question": "What is the difference between reduceByKey and groupByKey in Spark?"
    },
    # required
    # https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1771
    outputs=(
        "reduceByKey aggregates data before shuffling, whereas groupByKey "
        "shuffles all data, making reduceByKey more efficient."
    ),
    # expectations: dict[str, Any] | None = None,
    expectations={
        "expected_response": (
            "reduceByKey aggregates data before shuffling"
            "groupByKey shuffles all data"
        )
    },
)
print(f"assessment:\n{assessment}")

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

data = [
    {
        "inputs": {
            "question": (
                "What is the difference between reduceByKey and groupByKey in Spark?"
            )
        },
        "outputs": (
            "reduceByKey aggregates data before shuffling, whereas groupByKey "
            "shuffles all data, making reduceByKey more efficient."
        ),
        "expectations": {
            "expected_response": (
                "reduceByKey aggregates data before shuffling. groupByKey shuffles all data"
            ),
        },
    }
]
result = mlflow.genai.evaluate(data=data, scorers=[Correctness()])
print(result)

## Correctness scorer

Evaluates correctness of the response against expectations (_ground truth_)

`Correctness` is a built-in scorer ([BuiltInScorer](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L314-L389)).

Properties:
* `name` always **correctness**
* `model` (discussed later)
* `required_columns` are "inputs", "outputs"

Correctness is a LLM judge that uses a judge model for evaluation (discussed later).

In [0]:
# Part of BuiltInScorer.model_validate

from mlflow.genai.scorers import builtin_scorers

correctness_class = getattr(builtin_scorers, 'Correctness')
print(correctness_class)

In [0]:
from mlflow.genai.scorers import Correctness

# Note the Args section for the model argument (discussed later)
help(Correctness)


### LLM Judges

`Correctness` is one of the built-in scorers that accepts a LLM judge model for execution.

<br>

```
 |      model: Judge model to use. Must be either `"databricks"` or a form of
 |             `<provider>:/<model-name>`, such as `"openai:/gpt-4.1-mini"`,
 |             `"anthropic:/claude-3.5-sonnet-20240620"`. MLflow natively supports
 |             `["openai", "anthropic", "bedrock", "mistral"]`, and more providers are supported
 |             through `LiteLLM <https://docs.litellm.ai/docs/providers>`_.
 |             Default model depends on the ``MLFLOW_GENAI_JUDGE_DEFAULT_MODEL`` environment
 |             variable and the tracking URI setup:
 |
 |             * If ``MLFLOW_GENAI_JUDGE_DEFAULT_MODEL`` is set, that value is used.
 |             * Databricks tracking URI: `databricks`
 |             * Otherwise: `openai:/gpt-4.1-mini`.
```

Other LLM judge model-based built-in scorers (that use `mlflow.genai.judges.utils.invoke_judge_model` for execution):

* `Equivalence` for semantic equivalence.
* `InstructionsJudge` for trace evaluation based on user-provided instructions.
* `RetrievalRelevance` that measures whether each chunk is relevant to the input request. Uses natural language instructions to guide evaluation, making it flexible for various assessment criteria.
* Others that use the LLM judges (`mlflow/genai/judges/builtin.py` methods) and return `Feedback`:
    * `meets_guidelines`
    * `is_tool_call_efficient` (experimental since 3.8.0)
    * `is_tool_call_correct` (experimental since 3.8.0)
    * `is_safe`
    * `is_grounded`
    * `is_correct`
    * `is_context_sufficient`
    * `is_context_relevant`

In [0]:
# the default LLM judge model is used when no model is passed explicitly.
# model_uri is the full model URI (e.g., "openai:/gpt-4.1-mini", "databricks").
# - The default Databricks judge ("databricks")
# - AI Gateway (Databricks model serving) for model provider endpoints ("<provider>:/<model-name>")
# - LiteLLM-supported models (provided LiteLLM is installed)

from mlflow.genai.judges.utils import get_default_model

print(get_default_model())

In [0]:
%pip show litellm

In [0]:
from mlflow.genai.scorers import Correctness

help(Correctness.__call__)


`Correctness` returns a [mlflow.entities.assessment.Feedback](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment.py#L193-L363) (which is a [Assessment](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment.py#L34-L186) FWIW).

In [0]:
help(mlflow.entities.assessment.Feedback)


Among the properties of `Feedback` is `AssessmentSource` with [AssessmentSourceType](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment_source.py#L102C7-L194):

- `HUMAN`: Assessment performed by a human evaluator
- `LLM_JUDGE`: Assessment performed by an LLM-as-a-judge (e.g., GPT-4)
- `CODE`: Assessment performed by deterministic code/heuristics
- `SOURCE_TYPE_UNSPECIFIED`: Default when source type is not specified

In [0]:
yes_or_no = Correctness()

# builtin_scorer_class is required for built-in scorer deserialization using model_validate class method
assert yes_or_no.model_dump()['builtin_scorer_class'] == Correctness.__name__

display(yes_or_no.model_dump())

In [0]:
print(yes_or_no.instructions)

In [0]:
# Type of the feedback value

yes_or_no.feedback_value_type

In [0]:
# the input fields for this judge

input_fields = [
    {**field.__dict__, "value_type": str(field.value_type)}
    for field in yes_or_no.get_input_fields()
]
display(input_fields)

In [0]:
# Correctness.validate_columns validates the input columns (the input fields above)

# Checks required_columns first
# {"inputs", "outputs"}

print(f"required_columns: {yes_or_no.required_columns}")

columns: set[str] = set()
try:
    yes_or_no.validate_columns(columns)
except Exception as e:
    print(e)

# Then, Correctness-specific ones
# {"expectations/expected_response", "expectations/expected_facts"}

columns = {'outputs', 'inputs'}
try:
    yes_or_no.validate_columns(columns)
except Exception as e:
    print(e)


<br>

```py
def __call__(
    self,
    *,
    inputs: dict[str, Any] | None = None,
    outputs: Any | None = None,
    expectations: dict[str, Any] | None = None,
    trace: Trace | None = None,
) -> Feedback:
    ...
```

[Correctness.\_\_call__](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1827-L1891)

# InstructionsJudge

[InstructionsJudge](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/judges/instructions_judge/__init__.py#L61-L902) is a judge that evaluates traces based on user-provided instructions.

# Session-Level Scorers


Scorers are not session-level by default ([Scorer.is_session_level_scorer](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/base.py#L251-L257) property is `False`).

Child classes can override this property to return `True` or compute the value dynamically based on their configuration (e.g., [InstructionsJudge](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/judges/instructions_judge/__init__.py#L229-L231)).

In [0]:
from mlflow.genai.judges.instructions_judge import InstructionsJudge

# Instructions template must contain at least one variable
# (e.g., {{ inputs }}, {{ outputs }}, {{ trace }}, {{ expectations }}, or {{ conversation }}).
# When {{ conversation }} is included in the template, the judge is a session-level scorer.

instructions_judge = InstructionsJudge(name='default_name', instructions='no conversation. only {{ inputs }}')
assert not instructions_judge.is_session_level_scorer

instructions_judge = InstructionsJudge(name='default_name', instructions='contains {{ conversation }} and is hence a session-level scorer')
assert instructions_judge.is_session_level_scorer

# Aggregations

Among the properties of Scorers are `aggregations`.

<br>

```py
aggregations: list[_AggregationType] | None = None
```

Aggregations can be the literals (`"min", "max", "mean", "median", "variance", "p90"`) or custom functions (of `int`s or `float`s).

<br>

```py
_AggregationType: TypeAlias = (Literal["min", "max", "mean", "median", "variance", "p90"] | _AggregationFunc)
```

```py
_AggregationFunc: TypeAlias = Callable[[list[int | float]], float]
```


# Demo: Evaluating CrewAI

[MLflow and CrewAI]($./WIP MLflow and crewAI)

# Learning Resources

1. [LLM Judges and Scorers](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/)